### Author
Iker García Ferrero

Github: https://github.com/ikergarcia1996

# Example Of Use


```
# Initialize the grammar and read the rules from a file
g = Grammar('example_grammar1.txt')

# Parse a sentence
g.parse('astronomers saw stars with ears')

# Print the table used for parsing the sentence
g.print_parse_table()

# Get the list of trees generated for the sentence
trees = g.get_trees()

# Get the result of the production rule, VP, S, NP...
p = trees[0].get_type

# Get the left child of the production rule
l = trees[0].get_left

# Get the right child of the production rule
d = trees[0].get_right
```



 ## Expected output


```
Grammar file readed succesfully. Rules readed:
S --> NP VP
PP --> P NP
VP --> V NP
VP --> VP PP
NP --> NP PP
NP --> astronomers
NP --> ears
NP --> saw
V --> saw
NP --> telescope
NP --> stars
P --> with

Applied Rule: VP[2,2] --> V[1,2] NP[1,3]
Applied Rule: PP[2,4] --> P[1,4] NP[1,5]
Applied Rule: S[3,1] --> NP[1,1] VP[2,2]
Applied Rule: NP[3,3] --> NP[1,3] PP[2,4]
Applied Rule: VP[4,2] --> V[1,2] NP[3,3]
Applied Rule: VP[4,2] --> VP[2,2] PP[2,4]
Applied Rule: S[5,1] --> NP[1,1] VP[4,2]
Applied Rule: S[5,1] --> NP[1,1] VP[4,2]
----------------------------------------
The sentence IS accepted in the language
Number of possible trees: 2
----------------------------------------

-----------  ------------  ------  ------  ------
['S', 'S']
[]           ['VP', 'VP']
['S']        []            ['NP']
[]           ['VP']        []      ['PP']
['NP']       ['NP', 'V']   ['NP']  ['P']   ['NP']
astronomers  saw           stars   with    ears
-----------  ------------  ------  ------  ------
```


# Example of grammar file
```
S -> NP VP
PP -> P NP
VP -> V NP
VP -> VP PP
NP-> NP PP
NP -> astronomers
NP -> ears
NP -> saw
NP-> telescope
NP -> stars
P -> with
V -> saw
```





In [ ]:
class Dictlist(dict):

    '''
    Role: Stores the grammar rules, where the key is the non-terminal (like S, NP, VP)
    and the value is a list of production rules for that non-terminal.
    The Grammar class will call the Dictlist methods (like __setitem__)
    to add production rules for each non-terminal symbol.

    For a rule like S → NP VP, the Dictlist would store S as the
    key and the value would be the production rule NP VP
    '''
    def __setitem__(self, key, value):
        try:
            self[key]
        except KeyError:
            super(Dictlist, self).__setitem__(key, [])
        self[key].append(value)

# {'s':'np vp', }
class production_rule(object):
    '''
     Represents each individual production rule (e.g., S → NP VP), where:
     result: The non-terminal symbol that the rule expands to (e.g., S).
     p1 and p2: The two components (left and right) of the production rule.

    When a new production rule is applied (for a substring of the sentence),
     an instance of production_rule is created and added to the corresponding
      Cell using the add_production method.


    '''
    result = None
    p1 = None
    p2 = None

    #Parameters:
    #   Result: String
    #   p1: Production rule (left child of the production rule)
    #   p2: Production rule (right child of the production rule)
    def __init__(self,result,p1,p2):
        self.result = result # S
        self.p1 = p1 # NP
        self.p2 = p2 # VP
    # Define the __str__ method for converting production_rule objects to a string
    def __str__(self):
        if self.p1 is None and self.p2 is None:
            return f"{self.result} "  # For terminal rules
        elif self.p2 is None:
            return f"{self.result} → {self.p1}"  # For unary rules
        else:
            return f"{self.result} → {self.p1} {self.p2}"  # For binary rules
    #Returns the result of the production rule, VP, S, NP...
    @property
    def get_type(self):
        return self.result

    #Returns the left child of the production rule
    @property
    def get_left(self):
        return self.p1

    #Returns the right child of the production rule
    @property
    def get_right(self):
        return self.p2

class Cell(object):
    '''
      Cell stores multiple production_rule objects in its productions attribute.
      Each production_rule in the Cell represents a possible way to expand a
      non-terminal for that particular substring of the sentence.

      Grammar uses Cell to populate the CYK parse table with potential
      non-terminals for each substring of the sentence.
      As the parsing progresses, Grammar populates Cell
      objects for each substring, storing production rules for those substrings.

      '''

    productions = []


    #Parameters:
    #   Productions: List of production rules

    def __init__(self, productions=None):
        if productions is None:
            self.productions = []
        else:
            self.productions = productions

    def add_production(self, result,p1,p2):
        self.productions.append(production_rule(result,p1,p2))

    def set_productions(self, p):
        self.productions = p

    @property
    def get_types(self):
        types = []
        for p in self.productions:
            types.append(p.result)
        return types
    @property
    def get_rules(self):
        return self.productions


class Grammar(object):

    grammar_rules = Dictlist()
    parse_table = None
    length = 0
    tokens = []
    number_of_trees = 0

    #Parameters:
    #   Filename: file containing a grammar

    def __init__(self, filename): # starts from here-------------------------
        self.grammar_rules = Dictlist()
        self.parse_table = None
        self.length = 0
        for line in open(filename):
            a, b = line.split("->")
            self.grammar_rules[b.rstrip().strip()]=a.rstrip().strip()

        if len(self.grammar_rules) == 0:
            raise ValueError("No rules found in the grammar file")
        print('')
        print('Grammar file readed succesfully. Rules read:')
        self.print_rules()
        print('')

    #Print the production rules in the grammar

    def print_rules(self):
        for r in self.grammar_rules:
            for p in self.grammar_rules[r]:
                print(str(p) + ' --> ' + str(r))

    def apply_rules(self,t):
        try:
            return self.grammar_rules[t]
        except KeyError as r:
            return None

    #Parse a sentence (string) with the CYK algorithm
    def parse(self, sentence): #-------------------- parsing starts from here
      self.number_of_trees = 0
      self.tokens = sentence.split()
      self.length = len(self.tokens)

      if self.length < 1:
          raise ValueError("The sentence could not be read")

      # Create a parse table with Cell objects
      self.parse_table = [[Cell() for x in range(self.length - y)] for y in range(self.length)]

      # Process the first line (single tokens)
      print("Filling the first row of the CYK table:")
      '''

        t = "saw"                          # current word being processed
        x = 1                              # position of "saw" in the sentence
        r = ["NP", "V"]                    # returned by apply_rules("saw")
        w = "NP"                           # first iteration of: for w in r
        '''


      for x, t in enumerate(self.tokens): # all the tokens in the snetence, x index, t token
          r = self.apply_rules(t) # grammar rules that can produce the token t
          if r is None:
              raise ValueError("The word " + str(t) + " is not in the grammar")
          else:
              for w in r:
                  self.parse_table[0][x].add_production(w, production_rule(t, None, None), None) # store them in the table For example, if t is "cat", the Cell at position [0][x] will store the production rule N → cat
          # p1=None, p2=None marks it as a terminal with no children
          # Print the current string being processed and the updated table
          print(f"\nProcessing: {t}")
          self.print_parse_table()  # Print the current state of the CYK table

      # Run CYK-Parser for longer substrings
      print("\nRunning CYK Parser for substrings of increasing length:")
      for l in range(2, self.length + 1): # l = 1 corresponds to individual tokens (which are already processed in the first row of the parse table).
                                          #l = 2 represents substrings of length 2 (pairs of adjacent words), which is the next step.
          for s in range(1, self.length - l + 2):
              for p in range(1, l - 1 + 1):
                  t1 = self.parse_table[p - 1][s - 1].get_rules
                  t2 = self.parse_table[l - p - 1][s + p - 1].get_rules
                  print(f"\n t1 (left side rules): {[str(rule) for rule in t1]}")  # List of rules for the first part of the substring
                  print(f"t2 (right side rules): {[str(rule) for rule in t2]}")  # List of rules for the second part of the substring

                  for a in t1:
                      # print(f"\nProcessing rule from t1 (left side): {str(a)}")
                      # # Get the substring corresponding to 'a' (left part of the substring)
                      # t1_substring = ' '.join(self.tokens[s-1:s+l-1])
                      # print(f"t1 is being applied to substring: '{t1_substring}'")
                      for b in t2:
                          # print(f"Processing rule from t2 (right side): {str(b)}")

                          # # Get the substring corresponding to 'b' (right part of the substring)
                          # t2_substring = ' '.join(self.tokens[s+l-1:s+l-1+(l-1)])
                          # print(f"t2 is being applied to substring: '{t2_substring}'")

                          # # Calculate the combined substring
                          # substring = ' '.join(self.tokens[s-1:s+l-1])
                          # print(f"Applying rule to the substring: '{substring}'")
                          r = self.apply_rules(str(a.get_type) + " " + str(b.get_type))
                          print("applying rules for",str(a.get_type) + " " + str(b.get_type))
                          if r is not None:
                              for w in r:
                                  print( " w in r", w)
                                  # Print the applied rule for this combination
                                  print(f'Applied Rule: {str(w)} [{l},{s}] --> {str(a.get_type)} [{p},{s}] {str(b.get_type)} [{l - p},{s + p}]')

                                  # Add the new production to the cell
                                  self.parse_table[l - 1][s - 1].add_production(w, a, b)

                  # Print the current state of the CYK table after filling this cell
                  print(f"\nAfter filling cell [{l-1}, {s-1}] with substrings '{' '.join(self.tokens[s-1:s+l-1])}':")
                  self.print_parse_table()  # Print the current state of the CYK table

      # Check if any parse trees are found
      self.number_of_trees = len(self.parse_table[self.length - 1][0].get_types)
      if self.number_of_trees > 0:
          print("----------------------------------------")
          print('The sentence IS accepted in the language')
          print(f'Number of possible trees: {self.number_of_trees}')
          print("----------------------------------------")
      else:
          print("--------------------------------------------")
          print('The sentence IS NOT accepted in the language')
          print("--------------------------------------------")



    #Returns a list containing the parent of the possible trees that we can generate for the last sentence that have been parsed
    def get_trees(self):
        return self.parse_table[self.length-1][0].productions


    #@TODO
    def print_trees(self):
        pass

    #Print the CYK parse trable for the last sentence that have been parsed.
    def print_parse_table(self):
        try:
            from tabulate import tabulate
        except (ModuleNotFoundError,ImportError) as r:
            import subprocess
            import sys
            import logging
            logging.warning('To print the CYK parser table the Tabulate module is necessary, trying to install it...')
            subprocess.call([sys.executable, "-m", "pip", "install", 'tabulate'])

            try:
                from tabulate import tabulate
                logging.warning('The tabulate module has been instaled succesfuly!')

            except (ModuleNotFoundError,ImportError) as r:
                logging.warning('Unable to install the tabulate module, please run the command \'pip install tabulate\' in a command line')


        lines = []



        for row in reversed(self.parse_table):
            l = []
            for cell in row:
                l.append(cell.get_types)
            lines.append(l)

        lines.append(self.tokens)
        print('')
        print(tabulate(lines))
        print('')





In [3]:
%pip install -q google-colab

import os
from pathlib import Path

# replace the following path according to your Google Drive path
# %cd/gdrive/My Drive/Monash-FIT-S1-2022/Basic-CYK-Parser
if "COLAB_RELEASE_TAG" in os.environ:
	from google.colab import drive
	drive.mount('/content/drive')
	file_path = "/content/drive/MyDrive/FIT5127-LABS/Week4/example_grammar1.txt"
else:
	# Local Jupyter fallback (no google.colab required)
	file_path = str(Path("example_grammar1.txt").resolve())

ERROR: Could not find a version that satisfies the requirement google-colab (from versions: none)
ERROR: No matching distribution found for google-colab
Note: you may need to restart the kernel to use updated packages.


In [4]:
g = Grammar(file_path) #('example_grammar1.txt')
g.parse('astronomers saw stars with ears')
g.print_parse_table()
trees = g.get_trees()
p = trees[0].get_type
l = trees[0].get_left
d = trees[0].get_right
p = trees[1].get_type
l = trees[1].get_left
d = trees[1].get_right


Grammar file readed succesfully. Rules read:
S --> NP VP
PP --> P NP
VP --> V NP
VP --> VP PP
NP --> NP PP
NP --> astronomers
NP --> ears
NP --> saw
V --> saw
NP --> telescope
NP --> stars
P --> with

Filling the first row of the CYK table:

Processing: astronomers



-----------  ---  -----  ----  ----
[]
[]           []
[]           []   []
[]           []   []     []
['NP']       []   []     []    []
astronomers  saw  stars  with  ears
-----------  ---  -----  ----  ----


Processing: saw

-----------  -----------  -----  ----  ----
[]
[]           []
[]           []           []
[]           []           []     []
['NP']       ['NP', 'V']  []     []    []
astronomers  saw          stars  with  ears
-----------  -----------  -----  ----  ----


Processing: stars

-----------  -----------  ------  ----  ----
[]
[]           []
[]           []           []
[]           []           []      []
['NP']       ['NP', 'V']  ['NP']  []    []
astronomers  saw          stars   with  ears
-----------  -----------  ------  ----  ----


Processing: with

-----------  -----------  ------  -----  ----
[]
[]           []
[]           []           []
[]           []           []      []
['NP']       ['NP', 'V']  ['NP']  ['P']  []
astronomers  saw          stars  

In [ ]:
file_path_2 = "/content/drive/MyDrive/FIT5127-LABS/Week4/example_grammar2.txt"
g = Grammar(file_path_2)
g.parse('she eats a fish with a fork')
g.print_parse_table()
trees2 = g.get_trees()


Grammar file readed succesfully. Rules read:
S --> NP VP
VP --> VP PP
VP --> V NP
VP --> eats
V --> eats
PP --> P NP
NP --> Det N
NP --> she
P --> with
N --> fish
N --> fork
Det --> a

Filling the first row of the CYK table:

Processing: she

------  ----  --  ----  ----  --  ----
[]
[]      []
[]      []    []
[]      []    []  []
[]      []    []  []    []
[]      []    []  []    []    []
['NP']  []    []  []    []    []  []
she     eats  a   fish  with  a   fork
------  ----  --  ----  ----  --  ----


Processing: eats

------  -----------  --  ----  ----  --  ----
[]
[]      []
[]      []           []
[]      []           []  []
[]      []           []  []    []
[]      []           []  []    []    []
['NP']  ['VP', 'V']  []  []    []    []  []
she     eats         a   fish  with  a   fork
------  -----------  --  ----  ----  --  ----


Processing: a

------  -----------  -------  ----  ----  --  ----
[]
[]      []
[]      []           []
[]      []           []       []
[]      [

In [ ]:
g = Grammar(file_path_2)
g.parse('eats she fork a fish')
g.print_parse_table()
g.get_trees()


Grammar file readed succesfully. Rules read:
S --> NP VP
VP --> VP PP
VP --> V NP
VP --> eats
V --> eats
PP --> P NP
NP --> Det N
NP --> she
P --> with
N --> fish
N --> fork
Det --> a

Filling the first row of the CYK table:

Processing: eats

-----------  ---  ----  --  ----
[]
[]           []
[]           []   []
[]           []   []    []
['VP', 'V']  []   []    []  []
eats         she  fork  a   fish
-----------  ---  ----  --  ----


Processing: she

-----------  ------  ----  --  ----
[]
[]           []
[]           []      []
[]           []      []    []
['VP', 'V']  ['NP']  []    []  []
eats         she     fork  a   fish
-----------  ------  ----  --  ----


Processing: fork

-----------  ------  -----  --  ----
[]
[]           []
[]           []      []
[]           []      []     []
['VP', 'V']  ['NP']  ['N']  []  []
eats         she     fork   a   fish
-----------  ------  -----  --  ----


Processing: a

-----------  ------  -----  -------  ----
[]
[]           []
[]    

[]

In [ ]:
for s in range(0,3):
  print(s)

0
1
2
